# Stan ADVI Default Settings: Where They Can Go Wrong

Four changes away from Stan's ADVI defaults, alone and in combination, against
one shared "default" run:

- **a**: `grad_samples=100` (default is 1)
- **b**: `adapt_engaged=0`, `eta` left at its own raw default (1.0)
- **c**: `adapt_engaged=0`, `eta=0.1`
- **d**: `tol_rel_obj=1e-6` (default is 0.01)

`b` and `c` both set `adapt_engaged=0` -- `c` is `b` with a specific `eta`
picked instead of the untuned default, not an independent setting. So
whenever a combination would include both `b` and `c` together, `c` is used
by itself (their `adapt engaged=0` blocks can't both appear in one CmdStan
command line, and `c` already extends `b`) -- see the note in Section 3.

**Run lengths:** any comparison involving `d` runs from iteration 1 to
`MAX_ITERS = 50,000` (tracked every `TRACK_EVERY = 10` iterations); every
other comparison runs from 1 to `FAST_MAX_ITERS = 5,000` (tracked every
`FAST_TRACK_EVERY = 1` iteration, i.e. every single iteration). Stan's
variational method only ever writes final draws, so a trajectory is
reconstructed by rerunning the same seed at each tracked `iter=` prefix from
scratch -- at this density that's roughly 60,000 separate CmdStan calls
across all scenarios combined, so both scenario batches below run every
scenario in the batch **concurrently** (`parallel=True`) to keep wall-clock
time down.

**Rerun vs. reload:** the two batch cells are gated by `if RERUN:` (set in
the first cell, default `True`) and save every scenario's trajectory to
`stan_default_demos_output/cache/`; the loading cell and every section's
plotting cell always run and only read from that cache, so `RERUN = False`
re-renders everything from the last run without recomputing anything.

All plots use Plotly.


In [1]:
import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go

RERUN = False  # set False to re-render plots/tables from cache without recomputing


def find_repo_root():
    cwd = Path.cwd().resolve()
    for candidate in [cwd, *cwd.parents]:
        if (candidate / "modulars").exists():
            return candidate.resolve()
    raise FileNotFoundError("Could not locate repo root containing modulars/")


REPO_ROOT = find_repo_root()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

STAN_FILE = REPO_ROOT / "single_MC" / "1dgaussian_knownvar" / "random_restarts" / "stan_1dgaussian_knownvar.stan"
if not STAN_FILE.exists():
    raise FileNotFoundError(f"Expected Stan model at {STAN_FILE}")

if Path(sys.prefix).name != "stan3":
    raise RuntimeError(
        f"This notebook must run in the miniconda environment named stan3; "
        f"current sys.prefix is {sys.prefix!r}."
    )

STAN3_CMDSTAN = Path(sys.prefix).resolve() / "bin" / "cmdstan"
if not STAN3_CMDSTAN.exists():
    raise FileNotFoundError(f"Expected CmdStan at {STAN3_CMDSTAN}")
os.environ["CMDSTAN"] = str(STAN3_CMDSTAN)
from cmdstanpy import cmdstan_path, set_cmdstan_path
set_cmdstan_path(str(STAN3_CMDSTAN))

print(f"Repo root: {REPO_ROOT}")
print(f"Stan model: {STAN_FILE}")
print(f"CmdStan path: {cmdstan_path()}")
print(f"RERUN = {RERUN}")


Repo root: /Users/madelynandersen/Documents/VB Bakeoff/simpleVI
Stan model: /Users/madelynandersen/Documents/VB Bakeoff/simpleVI/single_MC/1dgaussian_knownvar/random_restarts/stan_1dgaussian_knownvar.stan
CmdStan path: /Users/madelynandersen/miniconda3/envs/stan3/bin/cmdstan
RERUN = False


/Users/madelynandersen/miniconda3/envs/stan3/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Model and data

Same 1D-Gaussian-known-variance model used throughout `single_MC/1dgaussian_knownvar/`,
so the true posterior is available in closed form for comparison. This is
cheap/deterministic (no Stan call), so it isn't gated by `RERUN`.


In [2]:
from modulars import gaussian_1d, gaussian_1d_posterior
from modulars.stan_rr_test import (
    run_stan_random_restarts, compile_stan_model, write_stan_data_json, stan_result_by_scenario,
)

mu_prior, sigma_prior = 0.0, 10.0
sigma_like = 1.0
n_samples = 100
mu_like = 3.0

data = gaussian_1d(n_samples, mu_like, sigma_like, seed=0).astype(float)
true_mu_post, true_sigma_post = gaussian_1d_posterior(
    data, mu_prior=mu_prior, sigma_prior=sigma_prior, sigma_like=sigma_like,
)

stan_data = {
    "N": len(data),
    "y": data,
    "sigma_like": sigma_like,
    "mu_prior": mu_prior,
    "sigma_prior": sigma_prior,
}
PARAM_COLUMNS = ["mu"]

print(f"true posterior: mu = {true_mu_post:.4f}, sigma = {true_sigma_post:.4f}")


true posterior: mu = 3.0595, sigma = 0.1000


## Shared machinery

`A_ARGS`..`D_ARGS` are the four settings; scenario names below combine them
with underscores (e.g. `"a_d"` = `grad_samples=100` + `tol_rel_obj=1e-6`).
`save_trajectory`/`load_trajectory` persist a scenario's `(iterations,
mu_traj, sd_traj)` triple to `stan_default_demos_output/cache/<name>.npz`.
`plot_mean_comparison`/`plot_std_comparison` each render one
default-vs-ablation Plotly line chart, for the variational mean and standard
deviation respectively.


In [3]:
BASE_SEED = 42
TOL_REL_OBJ_TIGHT = "1e-6"

A_ARGS = ("grad_samples=100",)
B_ARGS = ("adapt", "engaged=0")
C_ARGS = ("adapt", "engaged=0", "eta=0.1")
D_ARGS = (f"tol_rel_obj={TOL_REL_OBJ_TIGHT}",)

MAX_ITERS = 50_000
TRACK_EVERY = 10
MIN_ITER = 1

FAST_MAX_ITERS = 5_000
FAST_TRACK_EVERY = 1
FAST_MIN_ITER = 1

RESULTS_DIR = Path("stan_default_demos_output")
RESULTS_DIR.mkdir(exist_ok=True, parents=True)
CACHE_DIR = RESULTS_DIR / "cache"
CACHE_DIR.mkdir(exist_ok=True, parents=True)


def save_trajectory(name, iterations, mu_traj, sd_traj):
    np.savez(CACHE_DIR / f"{name}.npz", iterations=iterations, mu_traj=mu_traj, sd_traj=sd_traj)


def load_trajectory(name):
    cached = np.load(CACHE_DIR / f"{name}.npz")
    return cached["iterations"], cached["mu_traj"], cached["sd_traj"]


def run_scenario_batch(scenario_args, output_subdir, max_iters, track_every, min_iter, cache_prefix):
    """Run every {name: cmd_args} scenario in one call (parallelized across
    scenarios) and save each one's trajectory to the cache under
    f"{cache_prefix}_{name}".
    """
    result = run_stan_random_restarts(
        stan_file=STAN_FILE,
        data=stan_data,
        param_columns=PARAM_COLUMNS,
        output_dir=RESULTS_DIR / output_subdir,
        max_iters=max_iters,
        n_restarts=1,
        track_every=track_every,
        min_iter=min_iter,
        scenarios=[{"name": name, "cmd_args": cmd_args} for name, cmd_args in scenario_args.items()],
        seed_offset=BASE_SEED,
        parallel=True,
        keep_outputs=False,
        show_progress=True,
    )
    for name, (means, stds, iterations) in stan_result_by_scenario(result).items():
        save_trajectory(f"{cache_prefix}_{name}", iterations, means[0], stds[0])


def _plot_comparison(iterations, default_traj, ablation_traj, ablation_label, title, yaxis_title, true_value):
    # Colors match the convention used across the other 1dgaussian_knownvar
    # notebooks (modulars/plot_rr.py): blue for the default/single setting,
    # green for the adjusted/multi setting, red for the true/best-value line.
    fig = go.Figure()
    fig.add_trace(go.Scatter(
        x=iterations, y=default_traj, mode="lines", name="default", line=dict(color="blue", width=2),
    ))
    fig.add_trace(go.Scatter(
        x=iterations, y=ablation_traj, mode="lines", name=ablation_label, line=dict(color="green", width=2),
    ))
    fig.add_hline(y=true_value, line=dict(color="red", dash="dash"), annotation_text="true posterior value")
    fig.update_layout(
        title=title,
        xaxis_title="iteration",
        yaxis_title=yaxis_title,
        legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="left", x=0),
        margin=dict(t=80),
    )
    fig.show()
    return fig


def plot_mean_comparison(iterations, default_traj, ablation_traj, ablation_label, title):
    return _plot_comparison(
        iterations, default_traj, ablation_traj, ablation_label, title,
        yaxis_title="variational mean estimate (mu)", true_value=true_mu_post,
    )


def plot_std_comparison(iterations, default_traj, ablation_traj, ablation_label, title):
    return _plot_comparison(
        iterations, default_traj, ablation_traj, ablation_label, title,
        yaxis_title="variational sd estimate (sigma)", true_value=true_sigma_post,
    )


## Run all scenarios

Two batches, each run in one `run_stan_random_restarts` call so its
scenarios execute concurrently:

- **Slow batch** (`MAX_ITERS`, every comparison that touches `d`): `default`,
  `d`, `a_d`, `a_b_d`, `a_b_c_d` (using `c` instead of `b` inside `a_b_c_d`,
  see Section 3).
- **Fast batch** (`FAST_MAX_ITERS`, everything else): `default`, `a`, `b`,
  `c`, `a_b`, `a_c`, `a_b_c` (identical to `a_c`, see Section 3).

Both batches include their own `"default"` run at matching resolution, since
the fast batch's `track_every=1` is finer than the slow batch's
`track_every=10`.


In [4]:
if RERUN:
    slow_scenario_args = {
        "default": (),
        "d": D_ARGS,
        "a_d": A_ARGS + D_ARGS,
        "a_b_d": A_ARGS + B_ARGS + D_ARGS,
        # b and c both set adapt engaged=0; CmdStan rejects two "adapt" blocks
        # in one command, and c (adapt off + eta=0.1) already extends b (adapt
        # off, untuned eta), so a_b_c_d uses c in place of b here.
        "a_b_c_d": A_ARGS + C_ARGS + D_ARGS,
    }
    run_scenario_batch(
        slow_scenario_args, output_subdir="slow_batch",
        max_iters=MAX_ITERS, track_every=TRACK_EVERY, min_iter=MIN_ITER,
        cache_prefix="slow",
    )


In [5]:
if RERUN:
    fast_scenario_args = {
        "default": (),
        "a": A_ARGS,
        "b": B_ARGS,
        "c": C_ARGS,
        "a_b": A_ARGS + B_ARGS,
        "a_c": A_ARGS + C_ARGS,
        "a_b_c": A_ARGS + C_ARGS,  # same reasoning as a_b_c_d above: c stands in for b+c together
    }
    run_scenario_batch(
        fast_scenario_args, output_subdir="fast_batch",
        max_iters=FAST_MAX_ITERS, track_every=FAST_TRACK_EVERY, min_iter=FAST_MIN_ITER,
        cache_prefix="fast",
    )


In [6]:
SLOW_NAMES = ["default", "d", "a_d", "a_b_d", "a_b_c_d"]
FAST_NAMES = ["default", "a", "b", "c", "a_b", "a_c", "a_b_c"]

slow_traj = {name: load_trajectory(f"slow_{name}") for name in SLOW_NAMES}
fast_traj = {name: load_trajectory(f"fast_{name}") for name in FAST_NAMES}

LABELS = {
    "a": "grad_samples=100",
    "b": "adapt_engaged=0",
    "c": "adapt_engaged=0, eta=0.1",
    "d": f"tol_rel_obj={TOL_REL_OBJ_TIGHT}",
    "a_b": "grad_samples=100 + adapt_engaged=0",
    "a_c": "grad_samples=100 + adapt_engaged=0, eta=0.1",
    "a_d": f"grad_samples=100 + tol_rel_obj={TOL_REL_OBJ_TIGHT}",
    "a_b_c": "grad_samples=100 + adapt_engaged=0, eta=0.1  (= a_c, see Section 3)",
    "a_b_d": f"grad_samples=100 + adapt_engaged=0 + tol_rel_obj={TOL_REL_OBJ_TIGHT}",
    "a_b_c_d": f"grad_samples=100 + adapt_engaged=0, eta=0.1 + tol_rel_obj={TOL_REL_OBJ_TIGHT}",
}

print("loaded slow scenarios:", list(slow_traj.keys()))
print("loaded fast scenarios:", list(fast_traj.keys()))
for name in SLOW_NAMES:
    _, mu, sd = slow_traj[name]
    print(f"  slow  {name:12s} final mu = {mu[-1]:.4f}, final sd = {sd[-1]:.4f}")
for name in FAST_NAMES:
    _, mu, sd = fast_traj[name]
    print(f"  fast  {name:12s} final mu = {mu[-1]:.4f}, final sd = {sd[-1]:.4f}")
print(f"true posterior mu = {true_mu_post:.4f}, true posterior sd = {true_sigma_post:.4f}")


loaded slow scenarios: ['default', 'd', 'a_d', 'a_b_d', 'a_b_c_d']
loaded fast scenarios: ['default', 'a', 'b', 'c', 'a_b', 'a_c', 'a_b_c']
  slow  default      final mu = 3.0843, final sd = 0.1100
  slow  d            final mu = 3.0595, final sd = 0.1038
  slow  a_d          final mu = 3.0572, final sd = 0.1008
  slow  a_b_d        final mu = 3.0609, final sd = 0.1022
  slow  a_b_c_d      final mu = 3.0582, final sd = 0.1015
  fast  default      final mu = 3.0843, final sd = 0.1100
  fast  a            final mu = 3.0610, final sd = 0.0995
  fast  b            final mu = 2.9626, final sd = 0.1117
  fast  c            final mu = 3.0444, final sd = 0.1500
  fast  a_b          final mu = 3.0384, final sd = 0.0999
  fast  a_c          final mu = 3.0659, final sd = 0.1065
  fast  a_b_c        final mu = 3.0659, final sd = 0.1065
true posterior mu = 3.0595, true posterior sd = 0.1000


## Section 1: each setting alone vs. default

Four plots: `a`, `b`, `c` vs. default from iteration 1 to `FAST_MAX_ITERS`;
`d` vs. default from iteration 1 to `MAX_ITERS`.


In [7]:
_, default_fast_mu, default_fast_sd = fast_traj["default"]
for key in ["a", "b", "c"]:
    iterations, mu, sd = fast_traj[key]
    plot_mean_comparison(iterations, default_fast_mu, mu, LABELS[key], f"mean: default vs. {LABELS[key]} (1 to {FAST_MAX_ITERS})")
    plot_std_comparison(iterations, default_fast_sd, sd, LABELS[key], f"sd: default vs. {LABELS[key]} (1 to {FAST_MAX_ITERS})")

_, default_slow_mu, default_slow_sd = slow_traj["default"]
iterations, mu, sd = slow_traj["d"]
plot_mean_comparison(iterations, default_slow_mu, mu, LABELS["d"], f"mean: default vs. {LABELS['d']} (1 to {MAX_ITERS})")
plot_std_comparison(iterations, default_slow_sd, sd, LABELS["d"], f"sd: default vs. {LABELS['d']} (1 to {MAX_ITERS})")


## Section 2: `a` with each of `b`, `c`, `d` vs. default

Three plots: `a_b`, `a_c` vs. default from 1 to `FAST_MAX_ITERS`; `a_d` vs.
default from 1 to `MAX_ITERS`.


In [8]:
for key in ["a_b", "a_c"]:
    iterations, mu, sd = fast_traj[key]
    plot_mean_comparison(iterations, default_fast_mu, mu, LABELS[key], f"mean: default vs. {LABELS[key]} (1 to {FAST_MAX_ITERS})")
    plot_std_comparison(iterations, default_fast_sd, sd, LABELS[key], f"sd: default vs. {LABELS[key]} (1 to {FAST_MAX_ITERS})")

iterations, mu, sd = slow_traj["a_d"]
plot_mean_comparison(iterations, default_slow_mu, mu, LABELS["a_d"], f"mean: default vs. {LABELS['a_d']} (1 to {MAX_ITERS})")
plot_std_comparison(iterations, default_slow_sd, sd, LABELS["a_d"], f"sd: default vs. {LABELS['a_d']} (1 to {MAX_ITERS})")


## Section 3: `a` + `b` with each of `c` and `d` vs. default

Two plots: `a_b_c` vs. default from 1 to `FAST_MAX_ITERS`; `a_b_d` vs.
default from 1 to `MAX_ITERS`.

**Note:** `a_b_c`'s actual CmdStan settings are identical to `a_c` from
Section 2 -- `c` (adapt off, `eta=0.1`) already is `b` (adapt off) with a
specific `eta`, and CmdStan can't accept two `adapt` blocks in one command
line, so adding `b` on top of `a_c` has no effect beyond what `a_c` already
does. This plot is expected to reproduce `a_c`'s plot exactly.


In [9]:
iterations, mu, sd = fast_traj["a_b_c"]
plot_mean_comparison(iterations, default_fast_mu, mu, LABELS["a_b_c"], f"mean: default vs. {LABELS['a_b_c']} (1 to {FAST_MAX_ITERS})")
plot_std_comparison(iterations, default_fast_sd, sd, LABELS["a_b_c"], f"sd: default vs. {LABELS['a_b_c']} (1 to {FAST_MAX_ITERS})")

iterations, mu, sd = slow_traj["a_b_d"]
plot_mean_comparison(iterations, default_slow_mu, mu, LABELS["a_b_d"], f"mean: default vs. {LABELS['a_b_d']} (1 to {MAX_ITERS})")
plot_std_comparison(iterations, default_slow_sd, sd, LABELS["a_b_d"], f"sd: default vs. {LABELS['a_b_d']} (1 to {MAX_ITERS})")


## Section 4: all four combined vs. default

One plot, `a_b_c_d` vs. default, from iteration 1 to `MAX_ITERS = 50,000`.
As in Section 3, this uses `c` in place of `b` (see the note there) --
`a_b_c_d`'s actual settings are `grad_samples=100 + adapt_engaged=0,
eta=0.1 + tol_rel_obj=1e-6`.


In [10]:
iterations, mu, sd = slow_traj["a_b_c_d"]
plot_mean_comparison(iterations, default_slow_mu, mu, LABELS["a_b_c_d"], f"mean: default vs. {LABELS['a_b_c_d']} (1 to {MAX_ITERS})")
plot_std_comparison(iterations, default_slow_sd, sd, LABELS["a_b_c_d"], f"sd: default vs. {LABELS['a_b_c_d']} (1 to {MAX_ITERS})")
